# 14 – Phase 4: higher input resolution

**Question:** does feeding the network the images **upsampled** to 320 or 384 px help, compared with the native 256 px?

**Why it could help.** Upsampling adds no information: the new pixels are bilinear mixtures of the old ones. What changes is how much of that information the network can use. ConvNeXt's first layer cuts the image into 4×4 blocks and summarises each block into one vector; then it halves the resolution three more times:

| input | after the stem | stage 2 | stage 3 | stage 4 |
|---|---|---|---|---|
| 256 px | 64×64 | 32×32 | 16×16 | 8×8 |
| 320 px | 80×80 | 40×40 | 20×20 | 10×10 |
| 384 px | 96×96 | 48×48 | 24×24 | 12×12 |

- A spiral arm 2–3 px wide falls inside a single 4×4 block at 256 px, mixed with the surrounding sky. At 384 px the same arm spans 3–4.5 px and is split across more blocks, so the early layers can resolve it before the resolution is reduced.
- Our main errors are exactly about fine structure: tight ↔ loose spiral, and disturbed → loose spiral.
- The ImageNet-pretrained filters are also used to objects that fill much of the frame. Our galaxies are often small, so enlarging them brings them closer to that scale.

**Runs** (everything else identical: the reference configuration of notebook 13 = V4 without class weights and without lines/rectangles, 25 epochs, batch 32):

| run | input size | relative cost |
|---|---|---|
| `ref_256` | 256 (native) | 1× (≈ 16 min) |
| `ref_320` | 320 | ≈ 1.6× (≈ 25 min) |
| `ref_384` | 384 | ≈ 2.25× (≈ 35 min) |

`ref_256` is needed because the reference configuration has never been trained as such: notebook 13 trained V4 *with* class weights and lines (R0) and each of them removed *separately*.

**Same pre-processing everywhere.** Every image the network sees is upsampled in the same way:
- training images;
- both validations;
- the test.

The network learns its filters at one scale, so it must be evaluated at that same scale. In training, the random augmentations are applied **at the native 256 px, before** the upsampling. This way:
- a blur of σ = 1 px, the noise grain and the image framing mean the same thing at every input size;
- training images are altered the way the test images are (altered at 256 px, then upsampled by us).

**Decision rule.** Run-to-run noise is ≈ 0.005–0.01 (notebook 13), so a resolution is adopted only if it beats `ref_256` by **≳ 0.01 on both validations**, on most folds. A smaller gain does not justify 1.6–2.25× the training time.

**Time:** ≈ 1 h 20 min in total. Run it in `tmux`, from the **VS Code terminal** (which is inside Ubuntu):
```
tmux new -s phase4
cd ~/ANN-DL-Challenge/families/elia && source ../../.venv/bin/activate
jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 14_phase4_resolution.ipynb
```
Detach with `Ctrl+B`, release, then `D`; reattach with `tmux attach -t phase4`. Completed runs are skipped if the notebook is restarted.

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import f1_score

import train_lib as T
import common as C

plt.rcParams["figure.dpi"] = 80
SMOKE_TEST = False          # True = tiny run (few images, 1 epoch, fold 0) only to check that the code works
data = T.load_data(smoke_test=SMOKE_TEST)
print("device:", T.DEVICE, "| outputs:", T.OUT_DIR)

## 1. What upsampling does to an image

The same galaxy at 256, 320 and 384 px, with a zoom on its centre: the 64×64 central pixels of the original, i.e. the same region of the sky in all three.

**Check:**
- no new detail appears;
- the edges become smoother, because every new pixel is a mix of its neighbours;
- the structure now spans more pixels.

In [ ]:
idx = int(np.where(data["y"] == 6)[0][0])           # a tight spiral
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for j, s in enumerate([256, 320, 384]):
    x = T.to_float(data["X_clean"][idx:idx + 1], s)[0].permute(1, 2, 0).cpu().numpy()
    c, h = s // 2, int(32 * s / 256)
    axes[0, j].imshow(x); axes[0, j].set_title(f"{s} × {s} px")
    axes[1, j].imshow(x[c - h:c + h, c - h:c + h], interpolation="nearest")
    axes[1, j].set_title(f"centre: {2 * h} × {2 * h} pixels (same sky region)")
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout(); plt.show()

## 2. The experiments

In [ ]:
EXPERIMENTS = [
    T.reference_config(),
    T.reference_config(run_name="ref_320", note="reference at 320 px", img_size=320),
    T.reference_config(run_name="ref_384", note="reference at 384 px", img_size=384),
]
if SMOKE_TEST:
    for cfg in EXPERIMENTS:
        cfg.update(run_name=cfg["run_name"] + "_SMOKE", epochs=1, folds=[0], batch_size=16)

KEYS = ["run_name", "img_size", "epochs", "batch_size", "geo_p", "hue_p", "sat_p", "blur_p", "blur_sigma",
        "noise_p", "lines_p", "patch_p", "class_weight_power"]
pd.DataFrame(EXPERIMENTS)[KEYS]

In [ ]:
results = {}
for cfg in EXPERIMENTS:
    results[cfg["run_name"]] = T.run_experiment(cfg, data)

## 3. Comparison

Out-of-fold weighted F1 against `ref_256`. For context, the table also shows:
- `v4_repro` (R0 of notebook 13), i.e. the same recipe with class weights and lines/rectangles;
- `v4_fatemeh`;
- `e3_hue_blur`.

In [ ]:
RUNS = [cfg["run_name"] for cfg in EXPERIMENTS]
REF = RUNS[0]

table = T.compare(RUNS[1:] + ["v4_repro", "v4_fatemeh", "e3_hue_blur"], reference=REF)
table.style.format("{:.4f}").background_gradient(subset=["Δ clean", "Δ shifted"], cmap="RdYlGn", vmin=-0.02, vmax=0.02)

In [ ]:
# fold by fold, paired with ref_256 (same validation images): Δ = run − ref_256
def fold_f1(run):
    oof = pd.read_csv(T.OUT_DIR / run / "oof.csv")
    out = {}
    for val in ["clean", "shifted"]:
        P = oof[[f"p{c}_{val}" for c in range(C.N_CLASSES)]].to_numpy()
        out[val] = [f1_score(oof.label[oof.fold == k], P[oof.fold == k].argmax(1), average="weighted") for k in range(5)]
    return out

ref = fold_f1(REF)
rows = []
for run in RUNS[1:]:
    if not (T.OUT_DIR / run / "oof.csv").exists():
        continue
    f = fold_f1(run)
    for val in ["clean", "shifted"]:
        d = np.array(f[val]) - np.array(ref[val])
        rows.append({"run": run, "validation": val, **{f"fold {k}": d[k] for k in range(5)},
                     "mean Δ": d.mean(), "folds better": int((d > 0).sum())})
paired = pd.DataFrame(rows).set_index(["run", "validation"])
paired.style.format({**{f"fold {k}": "{:+.4f}" for k in range(5)}, "mean Δ": "{:+.4f}"}) \
      .background_gradient(subset=["mean Δ"], cmap="RdYlGn", vmin=-0.02, vmax=0.02)

## 4. Per class: do the fine-structure classes gain?

If resolution helps for the reason above, the gains should concentrate on:
- **tight spiral (6)**;
- **loose spiral (7)**;
- **disturbed (0)**;
- possibly **barred spiral (5)**.

They should not appear on the smooth classes.

In [ ]:
pc_clean, pc_shift = T.per_class(RUNS, "clean"), T.per_class(RUNS, "shifted")
delta = lambda t: t.sub(t[REF], axis=0).drop(columns=REF)
for name, t in [("CLEAN", pc_clean), ("SHIFTED", pc_shift)]:
    print(f"{name} validation: per-class F1 of {REF}, then Δ of each resolution")
    display(pd.concat([t[[REF]], delta(t)], axis=1).style.format("{:+.3f}").format("{:.3f}", subset=[REF])
            .background_gradient(subset=list(delta(t).columns), cmap="RdYlGn", vmin=-0.05, vmax=0.05))

In [ ]:
# confusion between the three spiral-like classes + disturbed (clean validation, OOF, rows = true class)
from sklearn.metrics import confusion_matrix
cls = [0, 5, 6, 7]
fig, axes = plt.subplots(1, len(RUNS), figsize=(5 * len(RUNS), 4.3))
for ax, run in zip(np.atleast_1d(axes), RUNS):
    oof = pd.read_csv(T.OUT_DIR / run / "oof.csv")
    pred = oof[[f"p{c}_clean" for c in range(C.N_CLASSES)]].to_numpy().argmax(1)
    cm = confusion_matrix(oof.label, pred, labels=list(range(C.N_CLASSES)))
    sub = cm[np.ix_(cls, cls)] / cm[cls].sum(1, keepdims=True).clip(min=1)
    ax.imshow(sub, cmap="Blues", vmin=0, vmax=1)
    for i in range(len(cls)):
        for j in range(len(cls)):
            ax.text(j, i, f"{sub[i, j]:.2f}", ha="center", va="center", color="white" if sub[i, j] > 0.5 else "black")
    names = [f"{c} {C.CLASSES[c]}" for c in cls]
    ax.set_xticks(range(len(cls)), names, rotation=30, ha="right", fontsize=8); ax.set_yticks(range(len(cls)), names, fontsize=8)
    ax.set_title(run)
plt.suptitle("Share of each true class (rows) predicted as each class (columns), restricted to 4 classes")
plt.tight_layout(); plt.show()

## 5. Training curves and time

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 3.8))
for name in RUNS:
    f = T.OUT_DIR / name / "history.csv"
    if not f.exists():
        continue
    h = pd.read_csv(f).groupby("epoch").mean(numeric_only=True)
    axes[0].plot(h.index, h.train_loss, label=name)
    axes[1].plot(h.index, h.f1_clean, label=name)
    axes[2].plot(h.index, h.f1_shifted, label=name)
for ax, t in zip(axes, ["train loss", "F1 clean (mean of folds, no TTA)", "F1 shifted (mean of folds, no TTA)"]):
    ax.set_title(t); ax.set_xlabel("epoch"); ax.legend(fontsize=8)
axes[1].set_ylim(0.7, 0.9); axes[2].set_ylim(0.7, 0.9)
plt.tight_layout(); plt.show()
print({name: f"{r['minutes']:.1f} min" for name, r in results.items()})

## 6. Decision

Rule: a resolution is adopted if it beats `ref_256` by **≳ 0.01 on both validations**, on most folds. Otherwise 256 stays, because the extra cost is not justified.

| run | F1 clean (Δ) | F1 shifted (Δ) | folds better (clean / shifted) | minutes |
|---|---|---|---|---|
| ref_256 | ___ | ___ | – | ___ |
| ref_320 | ___ | ___ | ___ | ___ |
| ref_384 | ___ | ___ | ___ | ___ |

- Do the gains concentrate on the spiral classes and disturbed? ___
- **Chosen resolution:** ___ → Kaggle submission of the chosen run (only if it passes the rule).